# Granite 4 3B

In [5]:
from langchain_ollama import ChatOllama
from uqlm import WhiteBoxUQ, BlackBoxUQ
import pandas as pd
from pylate import indexes, models, retrieve

def recuperar_documento(pregunta, model):
    
    df = pd.read_csv("UQAP/rag.csv")
    
    index = indexes.PLAID(
        index_folder="uqap_index",
        index_name="rag",
    )
    
    retriever = retrieve.ColBERT(index=index)
    
    queries_embeddings = model.encode(
        [pregunta],
        batch_size=32,
        is_query=True,
    )
    
    scores = retriever.retrieve(
        queries_embeddings=queries_embeddings,
        k=3,
    )

    contextos = ""
    for ide in scores[0]:
        i = int(ide["id"]) - 1
        contextos += "Título: "+ str(df.loc[i]["Título"]) + "\nContexto:" + str(df.loc[i]["Texto"]) + "\n"
    return contextos

llm = ChatOllama(model="granite4", logprobs=True, top_logprobs=5)

df = pd.read_csv("preguntas_revisado.csv")
preguntas = []
for index, row in df.iterrows():
    preguntas.extend(row.Preguntas.splitlines())

model = models.ColBERT(
    model_name_or_path="LiquidAI/LFM2-ColBERT-350M",
)
model.tokenizer.pad_token = model.tokenizer.eos_token

dataset = []
for pregunta in preguntas:
    texto = recuperar_documento(pregunta, model)
    dataset.append(f"""Dado el siguiente texto:
    {texto}
    Y la siguiente pregunta:
    {pregunta}
    Responde la pregunta de forma escueta y directa utilizando el texto proporiconado como contexto.
    """)
scorers = ["normalized_probability", "min_probability", # Single logprobs
           "min_token_negentropy", "mean_token_negentropy", "probability_margin", #Top logprobs
           "semantic_density", "monte_carlo_probability", # Sampled logprobs
          ]

wbuq = WhiteBoxUQ(llm=llm, scorers=scorers)
    
results = await wbuq.generate_and_score(prompts=dataset)
results.to_df().to_csv("Resultados/White_granite4.csv", index=False)

NameError: name 'prompts' is not defined

In [ ]:
scorers = ["noncontradiction", "semantic_negentropy", "cosine_sim", "bert_score"]

bbuq = BlackBoxUQ(llm=llm, scorers=scorers)
    
results = await bbuq.generate_and_score(prompts=dataset)
results.to_df().to_csv("Resultados/Black_granite4.csv", index=False)